In [ ]:
import pandas as pd
from datetime import datetime, timedelta

books = pd.read_csv("books.csv", encoding="utf-8-sig").set_index("book_id")
members = pd.read_csv("members.csv", encoding="utf-8-sig").set_index("member_id")

In [ ]:
#3.จำลองการยืมหนังสือ 350 รายการ
random.seed(1)

books = load_books("books.csv")
members = load_members("members.csv")

loans = []

start_date = datetime(2025, 1, 1)


for i in range(1, 351):

    available_books = [
        b for b in books
        if b.is_available
    ]

    if not available_books:

        available_books = books

        for b in available_books:
            b.return_book()

    book = random.choice(available_books)

    member = random.choice(members)

    borrow_date = random_borrow_date(start_date)

    loan_id = f"L{i:04d}"

    loan = BookLoan(
        loan_id,
        book,
        member,
        borrow_date
    )

    book.borrow()

    outcome = decide_return_outcome()

    return_date = calculate_return_date(
        loan.due_date,
        outcome
    )

    if return_date is not None:
        loan.mark_returned(return_date)

    loans.append(loan)


print(f"จำลองการยืมเสร็จแล้วทั้งหมด {len(loans)} รายการ")

In [ ]:
import time

def simulate_one_loan(loan_id, books, members, start_date, pause=0.0):
    """จำลองขั้นตอนทั้งหมดตอนสมาชิก 1 คนมายืมหนังสือ 1 เล่ม แล้วคืนค่า object BookLoan ที่สร้างเสร็จแล้ว
    """

    print("=" * 60)

    available_books = [b for b in books if b.is_available]
    if not available_books:
        available_books = books
        for b in available_books:
            b.return_book()

    book = random.choice(available_books)
    member = random.choice(members)

    print(f"📚 รายการยืม #{loan_id}: สมาชิก '{member.name}' ({member.member_type}) รหัส {member.member_id} "
      f"เดินมาที่เคาน์เตอร์ ต้องการยืมหนังสือ")
    time.sleep(pause)

    print(f"🔎 พนักงานค้นหาหนังสือว่าง เจอ: '{book.title}' (รหัส {book.book_id}, หมวด {book.category_name})")
    time.sleep(pause)

    borrow_date = random_borrow_date(start_date)
    loan = BookLoan(loan_id, book, member, borrow_date)
    book.borrow()

    print(f"📝 ระบบบันทึกการยืมเรียบร้อย (สถานะหนังสือว่าง = {book.is_available})")
    print(f"   วันที่ยืม: {borrow_date.strftime('%Y-%m-%d')} | "
          f"กำหนดคืน: {loan.due_date.strftime('%Y-%m-%d')} "
          f"({member.loan_period_days} วัน ตามประเภทสมาชิก '{member.member_type}')")
    time.sleep(pause)



    return_date = calculate_return_date(loan.due_date, outcome)

    if return_date is not None:
        loan.mark_returned(return_date)
        print(f"✅ คืนหนังสือแล้วเมื่อ {return_date.strftime('%Y-%m-%d')} "
              f"(สถานะหนังสือกลับเป็นว่าง = {book.is_available})")
    else:
        print("⏳ ยังไม่คืนหนังสือ (สถานะ: not_returned)")

    late_fee = loan.calculate_late_fee()
    print(f"🧾 สรุปรายการ #{loan_id}: {member.name} ยืม '{book.title}' | ค่าปรับ: {late_fee} บาท")

    return loan


# เรียกใช้ทดสอบ 1 ครั้ง ด้วยข้อมูลชุดแยกต่างหาก (ไม่กระทบข้อมูลที่จะใช้จำลอง 350 รายการจริงในขั้นตอนที่ 4 ด้านบน)
demo_books = load_books("books.csv")
demo_members = load_members("members.csv")

demo_loan = simulate_one_loan("DEMO-001", demo_books, demo_members, datetime(2025, 1, 1), pause=0.5)
